# Figure 16 -- gradient cost vs free-parameter count

Loads `bench/results/density_reconstruction/gradient_cost_vs_nparams.json`, produced by `bench/payoff_static/gradient_cost_vs_nparams.py`. No computation here.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] if pathlib.Path.cwd().name == "jaccpot_paper" else pathlib.Path.cwd()))

import numpy as np
import matplotlib.pyplot as plt

from examples.jaccpot_paper.common import jsonio, style

style.apply()
FIG_DIR = jsonio.RESULTS_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
art = jsonio.read_result("density_reconstruction/gradient_cost_vs_nparams.json")
cfg, data = art["config"], art["data"]
recs = [r for r in data["records"] if not r.get("failed")]
if not recs:
    raise SystemExit("gradient_cost_vs_nparams.json has no successful rows")
failed = [r for r in data["records"] if r.get("failed")]

# LATENCY-BOUND ROWS ARE NOT PLOTTED AS RATIOS. Below the point where the FMM
# becomes compute bound, a wall-clock ratio between the forward and the gradient
# reports which graph XLA fused into fewer kernels -- it came out BELOW ONE in
# the leaf-64 sweep, which is arithmetically impossible for a reverse pass. Such
# rows still carry the cost-flat-in-P statement, so they stay in the left panel
# and are excluded only from the ratio annotation.
compute_bound = [r for r in recs if not r.get("latency_bound")]

pos = sorted((r for r in recs if r["parameterization"] == "positions"),
             key=lambda r: r["num_free_parameters"])
par = sorted((r for r in recs if r["parameterization"] == "parametric"),
             key=lambda r: r["N"])

fig, axes = style.figure(width=style.TWO_COL, height=2.9, ncols=2)

# -- left: wall-clock against free-parameter count ------------------------- #
ax = axes[0]
P = [r["num_free_parameters"] for r in pos]
ax.plot(P, [r["forward_seconds"] for r in pos], marker=style.MARKERS[0],
        color=style.ENTITY["forward"], label="forward only")
ax.plot(P, [r["forward_backward_seconds"] for r in pos], marker=style.MARKERS[1],
        color=style.ENTITY["forward_backward"], label="forward + backward")

# The parametric arm at P = 7 is the same operator with a 450000x narrower
# pytree. It is drawn as points, not a curve: it is one parameter count.
if par:
    ax.scatter([r["num_free_parameters"] for r in par],
               [r["forward_backward_seconds"] for r in par],
               marker=style.MARKERS[3], s=18, zorder=5,
               facecolor="none", edgecolor=style.INK,
               label="parametric (P = %d), same N" % par[0]["num_free_parameters"])

# Finite differences, EXTRAPOLATED as (P + 1) forward evaluations from this
# run's own measured forward. Never measured; the label says so.
fd_P = np.array([1e1, 1e2, 1e3, 1e4, 1e5, 1e6, 1e7, 3e7])
ref = data["annotations"]["reference_forward_seconds"]
ax.plot(fd_P, (fd_P + 1.0) * ref, linestyle=":", color=style.INK_MUTED,
        label="finite differences, extrapolated")

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("free parameters $P$")
ax.set_ylabel("wall-clock per evaluation [s]")

# The annotation that IS the argument, in human units.
years = data["annotations"]["finite_difference_at"]["10000000"]["years"]
ax.annotate("$10^7$ parameters:\n%.1f yr by finite\ndifferences" % years,
            xy=(1e7, (1e7 + 1) * ref), xytext=(3e3, (1e7 + 1) * ref * 0.06),
            fontsize=6.0, color=style.INK,
            arrowprops={"arrowstyle": "->", "color": style.INK_MUTED, "lw": 0.6})
style.finish(ax, legend=True, legend_kwargs={"loc": "upper left", "fontsize": 5.6})

# -- right: the reverse-pass multiple, and memory -------------------------- #
ax = axes[1]
Ns = [r["N"] for r in compute_bound if r["parameterization"] == "positions"]
ratios = [r["backward_over_forward"] for r in compute_bound
          if r["parameterization"] == "positions"]
flops = [r["backward_over_forward_flops"] for r in compute_bound
         if r["parameterization"] == "positions"]
ax.plot(Ns, ratios, marker=style.MARKERS[0], color=style.ENTITY["forward_backward"],
        label="wall-clock")
if any(f is not None for f in flops):
    ax.plot([n for n, f in zip(Ns, flops) if f is not None],
            [f for f in flops if f is not None],
            marker=style.MARKERS[2], linestyle="--", color=style.INK_MUTED,
            label="XLA flop count")
ax.axhline(3.0, color=style.GRID, lw=0.8, zorder=0)
ax.text(Ns[0], 3.15, "3x", fontsize=5.6, color=style.INK_MUTED)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("$N$")
ax.set_ylabel("(forward + backward) / forward")

memory = [(r["N"], r.get("forward_backward_temp_bytes"))
          for r in pos if r.get("forward_backward_temp_bytes")]
if memory:
    twin = ax.twinx()
    twin.plot([m[0] for m in memory], [m[1] / 2**30 for m in memory],
              marker=style.MARKERS[4], linestyle="-.", color=style.CATEGORICAL[0],
              lw=0.9)
    twin.set_yscale("log")
    twin.set_ylabel("gradient temp memory [GiB]", color=style.CATEGORICAL[0])
    twin.tick_params(axis="y", colors=style.CATEGORICAL[0])
style.finish(ax, legend=True, legend_kwargs={"loc": "upper left", "fontsize": 5.6})

ceiling = ""
if failed:
    ceiling = "   single-device ceiling: N=%d failed (%s)" % (
        failed[0]["N"], failed[0].get("error_type", "?"))
    print("ceiling:", failed[0]["N"], failed[0].get("error_type"))

fig.tight_layout()
style.footer(fig, "%s, %s, order %d, theta %.2f, leaf %d, M=%d, mode=%s%s" % (
    art["meta"]["device_kind"], cfg["precision"], cfg["order"], cfg["theta"],
    cfg["leaf_size"], cfg["M"], cfg.get("mode", "?"), ceiling))
style.save(fig, str(FIG_DIR / "fig_16_gradient_cost_vs_nparams.pdf"))


## Caption


Cost of a reverse-mode gradient against the number of free source positions.
**Left:** wall-clock for one forward evaluation and for one
forward-plus-backward, against free-parameter count $P$. Both are flat in $P$ --
the open diamonds are the 7-parameter parametric model evaluated through the
*same* operator at the same $N$, and they land on the high-dimensional points,
so a change of $P$ by a factor of $4.5\times10^{5}$ changes the cost by about
1%. The dotted line is finite differences, **extrapolated** as $(P+1)$ forward
evaluations from this run's own measured forward time; it was not measured, and
one-sided rather than central differences are assumed so the baseline is not
inflated. **Right:** the reverse-pass multiple against $N$, by wall-clock and by
XLA's flop count, with the gradient's peak temporary memory on the second axis.
The multiple is close to 3 up to $N\sim10^{4}$ and grows to roughly 13 at
$N=10^{6}$: bounded, and independent of $P$, but *not* a small constant across
this range. Points at which the pipeline is launch-latency rather than compute
bound are excluded from the right panel -- there a wall-clock ratio measures
kernel fusion and not differentiation -- but retained on the left, where the
statement is about $P$. The single-device ceiling in the footer is measured, by
running until it failed.
